# Frequency

## Setup

In [8]:
# Cell 00: Colab Stuff
import os

DEVELOPMENT_MODE = False

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    os.system('pip install transformers==4.57.6 --quiet')
    os.system('pip install --upgrade numpy --quiet')
    print("Dependencies installed")
except ImportError:
    IN_COLAB = False

In [9]:
# Cell 0: Imports
import sys
import torch
from pathlib import Path

# Colab: files are in /content/
# Local: notebook is in notebooks/, project root is one level up
if Path('/content/data').exists():
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))

from transformer_lens import HookedTransformer
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/trishasalas/Repos/Research/tmlr


In [10]:
# Cell 1: Device check
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


## Frequency Analysis

Corpus frequency (Infini-gram) → trajectory classification (Spearman correlation).
No model needed — this is corpus-level analysis.

In [ ]:
# Cell 2: Frequency analysis — corpus counts + trajectory correlation
import importlib
import src.frequency
importlib.reload(src.frequency)
from src.frequency import run_frequency_analysis

freq_results = run_frequency_analysis(PROJECT_ROOT)
freq_results['freq_df']

In [ ]:
# Cell 3: Inspect Spearman results
freq_results['correlation']

## Token Competition Trace (exploratory)

Per-prompt investigation: what tokens compete at the decision point?
Requires a loaded model.

In [ ]:
# Cell 4: Load model for token competition trace
model_name = "gpt2-xl"
model = HookedTransformer.from_pretrained(model_name)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
# Cell 5: Token competition trace — the mechanistic half of the frequency hypothesis.
# Trace every compound that has a trajectory class, so competition (a high-freq
# competitor at the decision point) can be compared against regress vs. climb.
from src.frequency import COMPOUNDS, token_competition_trace, save_competition_trace

TRACE_COMPOUNDS = [
    "skip_link", "keyboard_navigation",                     # peak_regress
    "focus_indicator", "semantic_html", "closed_captions",  # never_emerges
    "screen_reader", "alt_text",                            # monotonic_climb
    "color_contrast",                                       # mixed
]
prompt_by_name = {name: prompt for name, _w1, _w2, prompt in COMPOUNDS}

traces = {}
for name in TRACE_COMPOUNDS:
    comp = token_competition_trace(model, prompt_by_name[name], top_k=10)
    save_competition_trace(comp, PROJECT_ROOT, model_name, name)
    traces[name] = comp
    print(f"\n{name}  —  {prompt_by_name[name]!r}")
    for tok, prob in comp["final_top"][:5]:
        print(f"    {tok!r:>15}  {prob:.4f}")

# Inspect any single trace, e.g. traces["skip_link"]["traces"]
traces["skip_link"]["traces"]

In [ ]:
# Cell 6: Ad hoc Infini-gram queries
from src.frequency import query_infinigram
query_infinigram("skip link")

### Delete Model & Clear Cache

In [ ]:
# Cell 7: Free memory
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")

In [12]:

import numpy as np
from scipy.stats import spearmanr
import pandas as pd

# Load existing data
freq = pd.read_csv('../results/frequency/frequency_table.csv')
coded = pd.read_csv('../results/analysis/elicitation_coded.csv')

score_map = {'correct': 1.0, 'partial': 0.5, 'incorrect': 0.0}
coded['score'] = coded['accuracy'].map(score_map)

for suite in ['pythia', 'gpt2']:
    s = coded[coded['suite'] == suite]
    by_concept = s.groupby('concept').agg(mean_accuracy=('score', 'mean')).reset_index()
    by_concept['compound'] = by_concept['concept'].str.replace(' ', '_').str.lower()
    
    merged = by_concept.merge(freq, on='compound', how='inner')
    merged['log_freq'] = np.log10(merged['bigram_count'].clip(lower=1))
    
    rho, p = spearmanr(merged['log_freq'], merged['mean_accuracy'])
    print(f"{suite}: ρ={rho:.4f}, p={p:.4f}, n={len(merged)}")

pythia: ρ=0.5289, p=0.0001, n=49
gpt2: ρ=0.4403, p=0.0015, n=49


In [13]:
spearman_results = pd.DataFrame([
    {'suite': 'pythia', 'corpus': 'pile', 'corpus_match': 'exact', 'rho': 0.5289, 'p': 0.0001, 'n': 49},
    {'suite': 'gpt2', 'corpus': 'pile', 'corpus_match': 'proxy', 'rho': 0.4403, 'p': 0.0015, 'n': 49},
    {'suite': 'olmo2-1b', 'corpus': 'dolma', 'corpus_match': 'exact', 'rho': 0.4776, 'p': 0.0005, 'n': 49},
])

spearman_results.to_csv(PROJECT_ROOT / 'results' / 'frequency' / 'spearman_accuracy.csv', index=False)
print(spearman_results)

      suite corpus corpus_match     rho       p   n
0    pythia   pile        exact  0.5289  0.0001  49
1      gpt2   pile        proxy  0.4403  0.0015  49
2  olmo2-1b  dolma        exact  0.4776  0.0005  49
